# Model Arena · API genérica ejecutable

Demostración con datos sintéticos de nivel y cambio: demuestra que el core no exige nombres contables. Ejecuta HPO pequeño y entrenamiento real. Los parámetros reducidos son exclusivamente de esta demo; no alteran la configuración productiva del notebook ADA.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tempfile import mkdtemp
from model_arena import ModelArena, ArenaSpec, PanelSchema, ModelSpec

dates = pd.date_range("2025-01-01", periods=35)
rows = []
for i, key in enumerate(["A", "B", "C"]):
    for t, date in enumerate(dates):
        rows.append(dict(series=key, date=date, level=100+i*10+t+np.sin(t), change=np.sin(t)))
history = pd.DataFrame(rows)
metadata = pd.DataFrame(dict(series=["A", "B", "C"], population=["seen", "seen", "zero_shot"], group=[1, 1, 1]))
workspace = Path(mkdtemp(prefix="arena_demo_"))
spec = ArenaSpec(
    schema=PanelSchema(id_col="series", time_col="date", target_cols=["level", "change"],
                       population_col="population", group_cols=["group"]),
    models=[ModelSpec(alias="history", params={})], h=2, window=4, freq="D", artifacts=str(workspace / "runs"),
    training=dict(epochs=2, final_epochs=3, batch_size=4, samples_per_epoch=8, cpu_threads=1),
    search=dict(n_trials=2, width=[8], depth=[1], decoder_depth=[1], encoder_design=["shared"], activation=["GELU"]),
)
arena = ModelArena(spec)
arena.plan(df=history, static_df=metadata)

In [ ]:
study = arena.tune(df=history, static_df=metadata)
curves = arena.results.learning_curves(scope="total")
selection = arena.select(study)
arena.fit(df=history, static_df=metadata, selection=selection, epochs=3)
forecasts = arena.predict()
artifact = arena.save(workspace / "fitted")
restored = ModelArena.load(artifact)
restored_forecasts = restored.predict(df=history, static_df=metadata)
np.testing.assert_allclose(forecasts.value, restored_forecasts.value, rtol=1e-6)
release = arena.postprocess(forecasts)
publication = arena.publish(release, destination=workspace / "publications")
print(publication.path)
forecasts.head()

In [ ]:
import plotly.express as px
px.line(curves, x="epoch", y="value", color="candidate", line_dash="split", facet_row="target", facet_col="fold",
        labels={"value": "Pinball normalizado", "epoch": "Época"})

Opcional: abre el scaffold genérico del dashboard. El monitor MAC que conserva el alertamiento contable se abre desde el notebook Fast.

In [ ]:
# from model_arena.observatory import Observatory
# from IPython.display import display, HTML
# monitor = Observatory(publication).launch()
# display(HTML(monitor.button()))